**PREPROCESSING DATA**

In [1]:
!pip install panda

  Preparing metadata (setup.py) ... done
  Created wheel for panda: filename=panda-0.3.1-py3-none-any.whl size=7239 sha256=db75005c4fd1ad2d7f01bff23627ef5dd41db8277127d01c1853a4ebc2e5e5c0
  Stored in directory: /root/.cache/pip/wheels/98/41/5b/6ca54e0b6a35e1b7248c12f56fcb753dfb7717fefaa0fb45f5
Successfully built panda


In [13]:
import pandas as pd
def load_data(filename):
  path = 'https://raw.githubusercontent.com/huynhhoc/DataAnalystDeepLearning/main/Data/covid19/'
  return pd.read_csv(path + filename)

Read file

In [14]:
df_csv = load_data('countries-aggregated_csv.csv')
print(df_csv.head())

        Date     CountryRegion  Confirmed  Recovered  Deaths
0  1/22/2020  Afghanistan-Asia        0.0        NaN     0.0
1  1/22/2020    Albania-Europe        0.0        NaN     0.0
2  1/22/2020    Algeria-Africa        0.0        NaN     0.0
3  1/22/2020    Andorra-Europe        0.0        NaN     0.0
4  1/22/2020     Angola-Africa        0.0        NaN     0.0


Convert the date to day amount which computer understands by the minimum date
when covid-19 occured. We have double-property "CountryRegion". The N/A value have to be set a default value (0.0)



In [15]:

df_csv['Date'] = pd.to_datetime(df_csv['Date'])
min_date = df_csv['Date'].min()
df_csv['Days_Since_Start'] = (df_csv['Date'] - min_date).dt.days

df_csv['Recovered'] = df_csv['Recovered'].fillna(0)
df_csv['Deaths'] = df_csv['Deaths'].fillna(0)
df_csv['Confirmed'] = df_csv['Confirmed'].fillna(0)

print(df_csv.head())

        Date     CountryRegion  Confirmed  Recovered  Deaths  Days_Since_Start
0 2020-01-22  Afghanistan-Asia        0.0        0.0     0.0                 0
1 2020-01-22    Albania-Europe        0.0        0.0     0.0                 0
2 2020-01-22    Algeria-Africa        0.0        0.0     0.0                 0
3 2020-01-22    Andorra-Europe        0.0        0.0     0.0                 0
4 2020-01-22     Angola-Africa        0.0        0.0     0.0                 0


Developer utilities to help to split column

In [16]:
import pandas as pd
def split_name_series(string):
  if isinstance(string, str) and '-' in string:
    country, region = string.split('-', 1)
  else:
    country = string
    region = None
  return pd.Series((country, region), index=['country','region'])

In [17]:
def split_name(x_df):
  res = x_df['CountryRegion'].apply(split_name_series)
  x_df[res.columns] = res
  return x_df

In [18]:
df_csv = split_name(df_csv)
print(df_csv.head())

        Date     CountryRegion  Confirmed  Recovered  Deaths  \
0 2020-01-22  Afghanistan-Asia        0.0        0.0     0.0   
1 2020-01-22    Albania-Europe        0.0        0.0     0.0   
2 2020-01-22    Algeria-Africa        0.0        0.0     0.0   
3 2020-01-22    Andorra-Europe        0.0        0.0     0.0   
4 2020-01-22     Angola-Africa        0.0        0.0     0.0   

   Days_Since_Start      country  region  
0                 0  Afghanistan    Asia  
1                 0      Albania  Europe  
2                 0      Algeria  Africa  
3                 0      Andorra  Europe  
4                 0       Angola  Africa  


Add the status label for each country

In [19]:
import sys
def create_evaluation_group(x_df):
  bins=[0.0,3000, 5000, sys.maxsize]
  labels=['normal','medium','worst']
  evaluationGroup=pd.cut(x_df['Confirmed'],bins=bins,labels=labels,include_lowest=True)
  x_df['Evaluation']=evaluationGroup
  return x_df

In [20]:
df_csv=(df_csv.pipe(create_evaluation_group))
df_csv

,Date,CountryRegion,Confirmed,Recovered,Deaths,Days_Since_Start,country,region,Evaluation
0,2020-01-22,Afghanistan-Asia,0.0,0.0,0.0,0,Afghanistan,Asia,normal
1,2020-01-22,Albania-Europe,0.0,0.0,0.0,0,Albania,Europe,normal
2,2020-01-22,Algeria-Africa,0.0,0.0,0.0,0,Algeria,Africa,normal
3,2020-01-22,Andorra-Europe,0.0,0.0,0.0,0,Andorra,Europe,normal
4,2020-01-22,Angola-Africa,0.0,0.0,0.0,0,Angola,Africa,normal
...,...,...,...,...,...,...,...,...,...
39475,2020-08-18,West Bank and Gaza-Unknown,17306.0,9939.0,113.0,209,West Bank and Gaza,Unknown,worst
39476,2020-08-18,Western Sahara-Unknown,10.0,8.0,1.0,209,Western Sahara,Unknown,normal
39477,2020-08-18,Yemen-Asia,1889.0,1052.0,537.0,209,Yemen,Asia,normal
39478,2020-08-18,Zambia-Africa,9981.0,8776.0,264.0,209,Zambia,Africa,worst


**NORMALIZE DATA**

In [31]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

In [32]:

X_raw = df_csv[['Confirmed', 'Recovered']].values
y_raw = df_csv['Deaths'].values.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2,
                                                    random_state=42)


feature_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_train)
X_test_scaled = feature_scaler.transform(X_test)

target_scaler = MinMaxScaler()
y_train_scaled = target_scaler.fit_transform(y_train)
y_test_scaled = target_scaler.transform(y_test)

X_train_scaled = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
X_test_scaled = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]

In [33]:
def gradient_descent(X, y, learning_rate=0.01, epochs=20):
    m = len(y)
    n_features = X.shape[1]
    weights = np.random.randn(n_features, 1)

    for epoch in range(epochs):
        y_pred = X.dot(weights)
        error = y_pred - y
        gradients = (1/m) * X.T.dot(error)
        weights -= learning_rate * gradients
        loss = np.mean(error**2)
    return weights
lr = 0.1
learned_weights = gradient_descent(X_train_scaled, y_train_scaled, learning_rate=lr, epochs=20)

In [34]:
y_pred_test_scaled = X_test_scaled.dot(learned_weights)
y_pred_test_real = target_scaler.inverse_transform(y_pred_test_scaled)
y_test_real = target_scaler.inverse_transform(y_test_scaled)
mse_real = mean_squared_error(y_test_real, y_pred_test_real)
r2_real = r2_score(y_test_real, y_pred_test_real)
print(f"Bộ trọng số tối ưu tìm được (w0, w1, w2):\n{learned_weights}")
print(f"Mean Squared Error (MSE thực tế): {mse_real:,.2f}")
print(f"R-squared (Độ chính xác của mô hình): {r2_real:.4f}")
print("="*40)

Bộ trọng số tối ưu tìm được (w0, w1, w2):
[[ 0.00497643]
 [-1.09241917]
 [ 1.16604283]]
Mean Squared Error (MSE thực tế): 7,335,018,736.14
R-squared (Độ chính xác của mô hình): -86.4855
